In [1]:
import pandas as pd
import plotly.express as px

from pacer.dat import read_dat_raw

In [2]:
# load_gps_files narrows each record to a GPSSample, whose timestamp_ms is the
# receiver's iTOW - GPS time of week, so reading it as a Unix epoch lands in
# January 1970. read_dat_raw keeps the PVT's own UTC fields instead.
df = read_dat_raw("/Volumes/SPEEDOR/pacer/SESS_044.dat")
df[["utc", "t", "lat", "lon", "speed_kmh", "numSV", "fixType", "hAcc_m"]]

,utc,t,lat,lon,speed_kmh,numSV,fixType,hAcc_m
0,2026-08-15 16:06:33.159765711,0.00,51.557389,-0.011814,0.9504,15,3,9.938
1,2026-08-15 16:06:33.199765685,0.04,51.557387,-0.011822,0.5976,16,3,9.319
2,2026-08-15 16:06:33.239765658,0.08,51.557384,-0.011831,0.5184,15,3,8.809
3,2026-08-15 16:06:33.279765631,0.12,51.557382,-0.011838,0.4680,16,3,8.363
4,2026-08-15 16:06:33.319765604,0.16,51.557380,-0.011843,0.3816,16,3,7.982
...,...,...,...,...,...,...,...,...
1430,2026-08-15 16:07:30.439725499,57.28,51.557465,-0.011847,0.6624,14,3,1.450
1431,2026-08-15 16:07:30.479725471,57.32,51.557465,-0.011847,0.6660,14,3,1.449
1432,2026-08-15 16:07:30.519725443,57.36,51.557466,-0.011848,0.6660,14,3,1.448
1433,2026-08-15 16:07:30.559725415,57.40,51.557466,-0.011848,0.7236,14,3,1.446


In [3]:
# Is it actually 25 Hz? Ask iTOW, which the receiver stamps itself: a 40 ms
# ladder is 25 Hz, and anything larger is a sample that never reached the log.
df["iTOW"].diff().value_counts()

iTOW
40.0    1432
80.0       2
Name: count, dtype: int64

In [4]:
df

,iTOW,year,month,day,hour,minute,sec,valid,tAcc,nano,...,flags3,headVeh,magDec,magAcc,boot_ms,utc,t,altitude,speed_kmh,hAcc_m
0,576411160,2026,8,15,16,6,33,247,1019,159765711,...,0,0,0,0,7530,2026-08-15 16:06:33.159765711,0.00,73.436,0.9504,9.938
1,576411200,2026,8,15,16,6,33,247,1018,199765685,...,0,0,0,0,7530,2026-08-15 16:06:33.199765685,0.04,72.520,0.5976,9.319
2,576411240,2026,8,15,16,6,33,247,1017,239765658,...,0,0,0,0,7539,2026-08-15 16:06:33.239765658,0.08,71.397,0.5184,8.809
3,576411280,2026,8,15,16,6,33,247,1017,279765631,...,0,0,0,0,7543,2026-08-15 16:06:33.279765631,0.12,70.645,0.4680,8.363
4,576411320,2026,8,15,16,6,33,247,1016,319765604,...,0,0,0,0,7544,2026-08-15 16:06:33.319765604,0.16,70.005,0.3816,7.982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1430,576468440,2026,8,15,16,7,30,247,1003,439725499,...,0,0,0,0,63350,2026-08-15 16:07:30.439725499,57.28,100.044,0.6624,1.450
1431,576468480,2026,8,15,16,7,30,247,1003,479725471,...,0,0,0,0,63414,2026-08-15 16:07:30.479725471,57.32,100.013,0.6660,1.449
1432,576468520,2026,8,15,16,7,30,247,1003,519725443,...,0,0,0,0,63419,2026-08-15 16:07:30.519725443,57.36,99.990,0.6660,1.448
1433,576468560,2026,8,15,16,7,30,247,1003,559725415,...,0,0,0,0,63429,2026-08-15 16:07:30.559725415,57.40,99.990,0.7236,1.446


In [5]:
# boot_ms is stamped by the app loop when it drains the receiver's queue, not
# by the receiver, so it is a burst-arrival clock rather than a sample clock -
# worth seeing against iTOW before trusting it for anything.
px.line(
    df.assign(lag_ms=lambda d: (d["boot_ms"] - d["boot_ms"].iloc[0]) - d["t"] * 1e3),
    x="t",
    y="lag_ms",
    title="Boot stamp drift against the receiver clock",
    labels={"t": "session time (s)", "lag_ms": "boot_ms - iTOW (ms)"},
)

In [6]:
df["utc"].diff().describe()

count                         1434
mean     0 days 00:00:00.040055759
std      0 days 00:00:00.001493305
min      0 days 00:00:00.039999970
25%      0 days 00:00:00.039999972
50%      0 days 00:00:00.039999972
75%      0 days 00:00:00.039999972
max      0 days 00:00:00.079999944
Name: utc, dtype: object